In [1]:
import os
import warnings
import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore")

WORKING_DIR = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/MP/results/"
os.makedirs(WORKING_DIR, exist_ok=True)
os.chdir(WORKING_DIR)

mpl.rcParams["pdf.fonttype"]       = 42
mpl.rcParams["ps.fonttype"]        = 42
sc.settings.set_figure_params(dpi=300, dpi_save=300)
mpl.rcParams["font.family"]        = "sans-serif"
mpl.rcParams["font.sans-serif"]    = ["Arial"]
mpl.rcParams["font.size"]          = 5
mpl.rcParams["font.weight"]        = "bold"
mpl.rcParams["axes.labelweight"]   = "bold"
mpl.rcParams["axes.titleweight"]   = "bold"
mpl.rcParams["figure.titleweight"] = "bold"
mpl.rcParams["axes.grid"]          = False
mpl.rcParams["axes.spines.top"]    = False
mpl.rcParams["axes.spines.right"]  = False
sns.set_style("ticks", {"axes.grid": False})
sns.set_style("ticks", {"axes.grid": False})

COND_ORDER = [
    "Control",
    "Prematurely-terminated",
    "3month-protocol",
    "6month-protocol",
    "PCMV/PRV",
    "Rejection",
]

COND_PALETTE = {
    "Control":                "#EFE48E",
    "3month-protocol":        "skyblue",
    "6month-protocol":        "#08528f",
    "Prematurely-terminated": "#C9B8D9",
    "PCMV/PRV":               "#F0C096",
    "Rejection":              "#E09E9E",
}

# --- A. Input / Output ---
GLOBAL_OBJ_PATH = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/merging_UMIs/Raw_UMI_merged_global_object_preclinical.h5ad"
RESULTS_DIR     = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/MP/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

CONDITION_COL  = "Condition"
SAMPLE_COL     = "Sample_ID"
CELL_TYPE_COL  = "cell_type"
CELL_STATE_COL = "cell_state"

BOXPLOT_FIGSIZE      = (4.25, 3.25)  # (width, height) of the boxplot figure
BOXPLOT_WIDTH        = 0.9         # Width of the paired boxes
BOXPLOT_LINEWIDTH    = 0.9         # Outline thickness of the box, whiskers, caps
MEDIAN_LINEWIDTH     = 0.7         # Thickness of the median line inside the box
STRIP_DOT_SIZE       = 4           # Size of the overlaid jitter dots
STRIP_ALPHA          = 0.6         # Transparency of the dots (0 to 1)
BRACKET_LINE_WIDTH   = 0.9         # Thickness of the significance bracket line
BRACKET_OFFSET_FRAC  = 0.02        # Gap above the max data point to start the bracket
BRACKET_TICK_FRAC    = 0.001       # Height of the downward bracket legs
STAR_GAP_FRAC        = 0.000001      # Gap between the bracket line and the asterisk
STAR_FONT_SIZE       = 7           # Size of the significance asterisks

# Visualization Parameters (Heatmap)
HEATMAP_FIGSIZE      = (5, 4)      # (width, height) of the heatmap figure
HEATMAP_CMAP         = "RdBu_r"    # Colormap for the heatmap
HEATMAP_ANNOT_SIZE   = 7           # Font size for the numbers inside the heatmap
DPI                  = 300         # Resolution for saved PDF files


# Gene set scoring

In [2]:

MYE_SUS_STATES = [
    "LYVE1_MP_sus",
    "SPP1_MP_sus",
    "AP_MP_sus",
    "ISG_MP_sus",
    "Prolif_Mye_sus",
    "PCMV_MP_sus",
]

MYE_PAPIO_STATES = [
    "AP_MP_papio",
    "Groundstate_MP_papio",
    "ISG_MP_papio",
    "SPP1_MP_papio",
    'GBE1_MP_papio', 
    'LYVE1_MP_papio'
]

ALL_MP_STATES = MYE_SUS_STATES + MYE_PAPIO_STATES

# --- E. Polarization pathways ---
PATHWAYS = {
    "Pro-inflammatory": [
        "BATF2", "STAT4", "IL23A", "IFNG", "GPR18", "CASP1",
        "TLR4", "IL6", "SOCS1", "CD38", "CD80", "CD86", "CD68",
        "FCGR1A", "CD40", "IRF1", "IRF8", "IRF5", "HIF1A", "STAT1",
        "JAK2", "PTEN", "CIITA", "PDE4B", "GBP3", "AIM2", "PFKFB3",
        "ALPK1", "EPSTI1", "TLR2", "NOS2", "IL1B",
        "CXCL9", "CXCL10", "CXCL11",
    ],
    "Anti-inflammatory": [
        "IL1RN", "VEGFA", "MYC", "AKT1", "CLEC10A", "STAT6", "IRF4",
        "GATA3", "TGM2", "RBPJ", "DAB2", "TGFBR1", "MTSS1", "CD163",
        "CD83", "EGR2", "IL10", "ARG1", "MRC1", "PPARG", "CCL18",
        "CCL17", "CCL24", "CCL8", "TGFB1", "TGFB2", "MERTK", "KLF4",
        "LAMTOR1", "PPARD", "IL4R",
    ],
}

# --- F. Gene expression filter threshold ---
EXPR_THRESHOLD = 0.0125    # Min mean expression in at least one cell state

# --- G. Pathway colors (for boxplots) ---
PATHWAY_PALETTE = {
    "Pro-inflammatory": "#B34DB3",  # Purple
    "Anti-inflammatory": "#4CAF50", # Green
}

# --- H. Significance test configuration ---
REF_COND = "3month-protocol"   # Reference condition for per-pathway stats
FDR_CUTOFF = 0.05              # FDR threshold for significance stars



# ==============================================================================
# 3. LOAD & PREPARE DATA
# ==============================================================================

print("Loading global object...")
adata = sc.read(GLOBAL_OBJ_PATH)
print(f"  Full object: {adata.shape[0]} cells x {adata.shape[1]} genes")

# Normalize
adata_gss = adata.copy()
sc.pp.normalize_total(adata_gss, target_sum=1e4)
sc.pp.log1p(adata_gss)

# --- Extract myeloid fractions ---
mye_mask = adata_gss.obs[CELL_TYPE_COL].isin(["Mye_sus", "Mye_papio"])
adata_mye = adata_gss[mye_mask].copy()

print(f"\n  Myeloid cells: {adata_mye.n_obs}")
print(f"  Cell types present: {adata_mye.obs[CELL_TYPE_COL].unique().tolist()}")
print(f"  Cell states present: {sorted(adata_mye.obs[CELL_STATE_COL].unique().tolist())}")

# --- Subset to MP states of interest ---
mp_mask = adata_mye.obs[CELL_STATE_COL].isin(ALL_MP_STATES)
adata_mp = adata_mye[mp_mask].copy()

print(f"\n  Macrophage cells after filtering: {adata_mp.n_obs}")
print(f"  Cell states kept: {sorted(adata_mp.obs[CELL_STATE_COL].unique().tolist())}")


Loading global object...
  Full object: 233200 cells x 40653 genes

  Myeloid cells: 13932
  Cell types present: ['Mye_sus', 'Mye_papio']
  Cell states present: ['AP_MP_papio', 'AP_MP_sus', 'DC_papio', 'DC_sus', 'GBE1_MP_papio', 'Groundstate_MP_papio', 'ISG_MP_papio', 'ISG_MP_sus', 'LYVE1_MP_papio', 'LYVE1_MP_sus', 'MO_papio', 'Mast_sus', 'PCMV_MP_sus', 'Phagocytic_MP_sus', 'Prolif_Mye_papio', 'Prolif_Mye_sus', 'SPP1_MP_papio', 'SPP1_MP_sus', 'VCAN_MO_sus']

  Macrophage cells after filtering: 11696
  Cell states kept: ['AP_MP_papio', 'AP_MP_sus', 'GBE1_MP_papio', 'Groundstate_MP_papio', 'ISG_MP_papio', 'ISG_MP_sus', 'LYVE1_MP_papio', 'LYVE1_MP_sus', 'PCMV_MP_sus', 'Prolif_Mye_sus', 'SPP1_MP_papio', 'SPP1_MP_sus']


In [3]:
print("\n--- Filtering genes by expression threshold ---")
filtered_pathways = {}

for pw_name, gene_list in PATHWAYS.items():
    genes_in_data = [g for g in gene_list if g in adata_mp.var_names]
    genes_missing = [g for g in gene_list if g not in adata_mp.var_names]

    if genes_missing:
        print(f"  [{pw_name}] Not in var_names ({len(genes_missing)}): {', '.join(genes_missing)}")

    if not genes_in_data:
        print(f"  [{pw_name}] No genes found. Skipping.")
        continue

    # Check expression per cell state
    expr_df = sc.get.obs_df(adata_mp, keys=genes_in_data + [CELL_STATE_COL])
    max_mean = expr_df.groupby(CELL_STATE_COL, observed=True).mean().max()

    retained = max_mean[max_mean >= EXPR_THRESHOLD].index.tolist()
    removed  = max_mean[max_mean < EXPR_THRESHOLD].index.tolist()

    if removed:
        print(f"  [{pw_name}] Below threshold ({len(removed)}): {', '.join(removed)}")
    else:
        print(f"  [{pw_name}] All genes retained.")

    if retained:
        filtered_pathways[pw_name] = retained

# Score
print("\nScoring pathways...")
for pw_name, gene_list in filtered_pathways.items():
    sc.tl.score_genes(adata_mp, gene_list=gene_list, score_name=f"{pw_name}_score")

# Save scored object
scored_path = os.path.join(RESULTS_DIR, "mp_polarization_scored.h5ad")
adata_mp.write(scored_path)
print(f"Saved scored object: {scored_path}")



--- Filtering genes by expression threshold ---
  [Pro-inflammatory] Below threshold (2): IL23A, IFNG
  [Anti-inflammatory] Below threshold (3): CLEC10A, GATA3, CCL24

Scoring pathways...
Saved scored object: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/MP/results/mp_polarization_scored.h5ad


# Global view data

In [4]:

import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


# Comment out any condition below to exclude it from the analysis
TARGET_CONDITIONS = [
    "Control",
    # "Prematurely-terminated",
    "3month-protocol",
    # "6month-protocol",
    # "PCMV/PRV",
    "Rejection",
]

TARGET_PATHWAYS = ["Pro-inflammatory", "Anti-inflammatory"]

PATHWAY_PALETTE = {
    "Pro-inflammatory": "#B34DB3",  # Purple
    "Anti-inflammatory": "#4CAF50", # Green
}

# Visualization Parameters (Boxplot)
BOXPLOT_FIGSIZE      = (4.2, 3.2)  # (width, height) of the boxplot figure
BOXPLOT_WIDTH        = 0.9         # Width of the paired boxes
BOXPLOT_LINEWIDTH    = 0.9         # Outline thickness of the box, whiskers, caps
MEDIAN_LINEWIDTH     = 0.7         # Thickness of the median line inside the box
STRIP_DOT_SIZE       = 4           # Size of the overlaid jitter dots
STRIP_ALPHA          = 0.6         # Transparency of the dots (0 to 1)
BRACKET_LINE_WIDTH   = 0.9         # Thickness of the significance bracket line
BRACKET_OFFSET_FRAC  = 0.02        # Gap above the max data point to start the bracket
BRACKET_TICK_FRAC    = 0.001       # Height of the downward bracket legs
STAR_GAP_FRAC        = 0.0001      # Gap between the bracket line and the asterisk
STAR_FONT_SIZE       = 8           # Size of the significance asterisks

# Visualization Parameters (Heatmap)
HEATMAP_FIGSIZE      = (5, 4)      # (width, height) of the heatmap figure
HEATMAP_CMAP         = "RdBu_r"    # Colormap for the heatmap
HEATMAP_ANNOT_SIZE   = 7           # Font size for the numbers inside the heatmap
DPI                  = 300         # Resolution for saved PDF files


# Global overview plot
overview_dir = os.path.join(RESULTS_DIR, "global_overview")
os.makedirs(overview_dir, exist_ok=True)

# Extract relevant columns from the global MP object using TARGET_PATHWAYS
df_global = adata_mp.obs[[f"{p}_score" for p in TARGET_PATHWAYS] + [SAMPLE_COL, CONDITION_COL]].copy()
df_global.columns = [c.replace("_score", "") for c in df_global.columns]

# Filter down to only the active conditions
df_global = df_global[df_global[CONDITION_COL].isin(TARGET_CONDITIONS)]

# Pseudobulk: median per sample across ALL macrophage states combined
grouped_global = df_global.groupby([CONDITION_COL, SAMPLE_COL], observed=True)[TARGET_PATHWAYS].median().dropna(how='all').reset_index()

# Save sample-level aggregated data
grouped_global.round(4).to_csv(os.path.join(overview_dir, "global_sample_median_scores.csv"), index=False)

# Melt for plotting and stats
melted_global = grouped_global.melt(
    id_vars=[CONDITION_COL, SAMPLE_COL],
    value_vars=TARGET_PATHWAYS, var_name="Pathway", value_name="Score"
)

global_stats = []
global_pvals = []
valid_conds = [c for c in TARGET_CONDITIONS if c in melted_global[CONDITION_COL].unique()]

for cond in valid_conds:
    cond_data = melted_global[melted_global[CONDITION_COL] == cond]
    pro  = cond_data[cond_data["Pathway"] == "Pro-inflammatory"]["Score"].dropna()
    anti = cond_data[cond_data["Pathway"] == "Anti-inflammatory"]["Score"].dropna()

    med_pro  = pro.median() if len(pro) > 0 else np.nan
    med_anti = anti.median() if len(anti) > 0 else np.nan
    log2fc   = np.log2(med_pro / med_anti) if med_anti > 0 and med_pro > 0 else np.nan

    pval = np.nan
    if len(pro) >= 2 and len(anti) >= 2:
        try:
            _, pval = mannwhitneyu(pro, anti, alternative="two-sided")
        except Exception:
            pass

    global_stats.append({
        "Condition": cond, "N_Samples_Pro": len(pro), "N_Samples_Anti": len(anti),
        "Median_Pro": round(med_pro, 4) if not np.isnan(med_pro) else np.nan,
        "Median_Anti": round(med_anti, 4) if not np.isnan(med_anti) else np.nan,
        "log2FC_Pro_vs_Anti": round(log2fc, 4) if not np.isnan(log2fc) else np.nan,
        "pvalue": pval,
    })
    if not np.isnan(pval):
        global_pvals.append(pval)

# Apply BH FDR correction
stats_df = pd.DataFrame(global_stats)
stats_df["FDR"] = np.nan

if global_pvals:
    valid_mask = stats_df["pvalue"].notna()
    _, fdr_vals, _, _ = multipletests(stats_df.loc[valid_mask, "pvalue"], method="fdr_bh")
    stats_df.loc[valid_mask, "FDR"] = fdr_vals

stats_df["Significance"] = stats_df["FDR"].apply(
    lambda f: "***" if f < 0.001 else ("**" if f < 0.01 else ("*" if f < 0.05 else "")) if not np.isnan(f) else ""
)
stats_df.to_csv(os.path.join(overview_dir, "global_polarization_stats.csv"), index=False)

# Making the boxplot that I want to use later
boxprops     = dict(edgecolor="black", linewidth=BOXPLOT_LINEWIDTH)
medianprops  = dict(color="black", linewidth=MEDIAN_LINEWIDTH)
whiskerprops = dict(color="black", linewidth=BOXPLOT_LINEWIDTH)
capprops     = dict(color="black", linewidth=BOXPLOT_LINEWIDTH)

fig, ax = plt.subplots(figsize=BOXPLOT_FIGSIZE)
sns.boxplot(
    data=melted_global, x=CONDITION_COL, y="Score", hue="Pathway",
    order=valid_conds, hue_order=TARGET_PATHWAYS, palette=PATHWAY_PALETTE,
    width=BOXPLOT_WIDTH, showfliers=False, boxprops=boxprops, medianprops=medianprops,
    whiskerprops=whiskerprops, capprops=capprops, ax=ax, dodge=True
)
sns.stripplot(
    data=melted_global, x=CONDITION_COL, y="Score", hue="Pathway",
    order=valid_conds, hue_order=TARGET_PATHWAYS, dodge=True, 
    palette=["black", "black"], size=STRIP_DOT_SIZE, alpha=STRIP_ALPHA, linewidth=0, legend=False, ax=ax
)

for _, row in stats_df.iterrows():
    if row["Significance"] == "": continue
    if row["Condition"] not in valid_conds: continue
    i = valid_conds.index(row["Condition"])

    cond_scores = melted_global[melted_global[CONDITION_COL] == row["Condition"]]["Score"]
    y_range = cond_scores.max() - cond_scores.min() if (cond_scores.max() - cond_scores.min()) > 0 else 1.0
    y = cond_scores.max() + BRACKET_OFFSET_FRAC * y_range
    tick_h = BRACKET_TICK_FRAC * y_range

    x1, x2 = i - 0.2, i + 0.2
    ax.plot([x1, x1, x2, x2], [y, y + tick_h, y + tick_h, y], lw=BRACKET_LINE_WIDTH, color="black")
    ax.text(i, y + tick_h + STAR_GAP_FRAC * y_range, row["Significance"], ha="center", va="bottom", fontsize=STAR_FONT_SIZE, fontweight="bold")

ax.set_title("Global MP Polarization", fontweight="bold", fontsize=9)
ax.set_xlabel("Condition", fontweight="bold", fontsize=8)
ax.set_ylabel("Score (Sample Median)", fontweight="bold", fontsize=8)
plt.xticks(rotation=45, ha="right", fontweight="bold", fontsize=7)
plt.yticks(fontweight="bold", fontsize=7)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], bbox_to_anchor=(1.05, 1), loc="upper left", frameon=False, prop={"weight": "bold", "size": 7}, title="Pathway")
plt.tight_layout()
plt.savefig(os.path.join(overview_dir, "global_boxplot.pdf"), dpi=DPI, bbox_inches="tight")
plt.close()

# Plotting global heatmaps
ht = melted_global.groupby([CONDITION_COL, "Pathway"], observed=True)["Score"].median().unstack("Pathway")
ht = ht.reindex([c for c in TARGET_CONDITIONS if c in ht.index])
ht.round(4).to_csv(os.path.join(overview_dir, "global_heatmap_data.csv"))

fig, ax = plt.subplots(figsize=HEATMAP_FIGSIZE)
sns.heatmap(ht, cmap=HEATMAP_CMAP, annot=True, fmt=".3f", linewidths=0.5, annot_kws={"weight": "bold", "size": HEATMAP_ANNOT_SIZE}, ax=ax)
ax.set_title("Global MP Polarization Heatmap", fontweight="bold", fontsize=9)
plt.xticks(rotation=30, ha="right", fontweight="bold", fontsize=8)
plt.yticks(rotation=0, fontweight="bold", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(overview_dir, "global_heatmap.pdf"), dpi=DPI, bbox_inches="tight")
plt.close()



# Per cell_state

In [5]:
# Comment out any condition below to exclude it from the analysis
TARGET_CONDITIONS = [
    "Control",
    # "Prematurely-terminated",
    "3month-protocol",
    # "6month-protocol",
    # "PCMV/PRV",
    "Rejection",
]

TARGET_PATHWAYS = ["Pro-inflammatory", "Anti-inflammatory"]

PATHWAY_PALETTE = {
    "Pro-inflammatory": "#B34DB3",  # Purple
    "Anti-inflammatory": "#4CAF50", # Green
}

# Visualization Parameters (Boxplot)
BOXPLOT_FIGSIZE      = (3.1, 3.5)  # (width, height) of the boxplot figure
BOXPLOT_WIDTH        = 0.9         # Width of the paired boxes
BOXPLOT_LINEWIDTH    = 0.9         # Outline thickness of the box, whiskers, caps
MEDIAN_LINEWIDTH     = 0.7         # Thickness of the median line inside the box
STRIP_DOT_SIZE       = 4           # Size of the overlaid jitter dots
STRIP_ALPHA          = 0.6         # Transparency of the dots (0 to 1)
BRACKET_LINE_WIDTH   = 0.9         # Thickness of the significance bracket line
BRACKET_OFFSET_FRAC  = 0.02        # Gap above the max data point to start the bracket
BRACKET_TICK_FRAC    = 0.001       # Height of the downward bracket legs
STAR_GAP_FRAC        = 0.0001      # Gap between the bracket line and the asterisk
STAR_FONT_SIZE       = 8           # Size of the significance asterisks

# Visualization Parameters (Heatmap)
HEATMAP_FIGSIZE      = (5, 4)      # (width, height) of the heatmap figure
HEATMAP_CMAP         = "RdBu_r"    # Colormap for the heatmap
HEATMAP_ANNOT_SIZE   = 7           # Font size for the numbers inside the heatmap
DPI                  = 300         # Resolution for saved PDF files


cellstate_dir = os.path.join(RESULTS_DIR, "per_cellstate")
os.makedirs(cellstate_dir, exist_ok=True)

# FIXED ERROR: Use TARGET_PATHWAYS directly instead of the undefined 'score_cols'
df_cs = adata_mp.obs[[f"{p}_score" for p in TARGET_PATHWAYS] + [SAMPLE_COL, CONDITION_COL, CELL_STATE_COL]].copy()
df_cs.columns = [c.replace("_score", "") for c in df_cs.columns]

# Filter down to only the active conditions
df_cs = df_cs[df_cs[CONDITION_COL].isin(TARGET_CONDITIONS)]

grouped = df_cs.groupby([CONDITION_COL, CELL_STATE_COL, SAMPLE_COL], observed=True)[TARGET_PATHWAYS].median().reset_index()

melted_cs = grouped.melt(
    id_vars=[CONDITION_COL, CELL_STATE_COL, SAMPLE_COL],
    value_vars=TARGET_PATHWAYS, var_name="Pathway", value_name="Score"
)

all_stats = []

for cs in sorted(adata_mp.obs[CELL_STATE_COL].unique()):
    cs_data = melted_cs[melted_cs[CELL_STATE_COL] == cs].copy()
    if cs_data.empty:
        continue

    safe_cs = cs.replace("/", "_").replace(" ", "_")
    cs_out  = os.path.join(cellstate_dir, safe_cs)
    os.makedirs(cs_out, exist_ok=True)

    cs_conds = [c for c in TARGET_CONDITIONS if c in cs_data[CONDITION_COL].unique()]

    # --- STATISTICAL TESTING ---
    cs_stats = []
    cs_pvals = []

    for cond in cs_conds:
        cond_data = cs_data[cs_data[CONDITION_COL] == cond]
        pro  = cond_data[cond_data["Pathway"] == "Pro-inflammatory"]["Score"].dropna()
        anti = cond_data[cond_data["Pathway"] == "Anti-inflammatory"]["Score"].dropna()

        med_pro  = pro.median() if len(pro) > 0 else np.nan
        med_anti = anti.median() if len(anti) > 0 else np.nan
        log2fc   = np.log2(med_pro / med_anti) if med_anti > 0 and med_pro > 0 else np.nan

        pval = np.nan
        if len(pro) >= 2 and len(anti) >= 2:
            try:
                _, pval = mannwhitneyu(pro, anti, alternative="two-sided")
            except Exception:
                pass

        cs_stats.append({
            "Cell_State": cs, "Condition": cond,
            "N_Pro": len(pro), "N_Anti": len(anti),
            "Median_Pro": round(med_pro, 4) if not np.isnan(med_pro) else np.nan,
            "Median_Anti": round(med_anti, 4) if not np.isnan(med_anti) else np.nan,
            "log2FC_Pro_vs_Anti": round(log2fc, 4) if not np.isnan(log2fc) else np.nan,
            "pvalue": pval,
        })
        if not np.isnan(pval):
            cs_pvals.append(pval)

    cs_stats_df = pd.DataFrame(cs_stats)
    cs_stats_df["FDR"] = np.nan
    if cs_pvals:
        valid_mask = cs_stats_df["pvalue"].notna()
        _, fdr_vals, _, _ = multipletests(cs_stats_df.loc[valid_mask, "pvalue"], method="fdr_bh")
        cs_stats_df.loc[valid_mask, "FDR"] = fdr_vals

    cs_stats_df["Significance"] = cs_stats_df["FDR"].apply(
        lambda f: "***" if f < 0.001 else ("**" if f < 0.01 else ("*" if f < 0.05 else "")) if not np.isnan(f) else ""
    )
    cs_stats_df.to_csv(os.path.join(cs_out, f"{safe_cs}_stats.csv"), index=False)
    all_stats.append(cs_stats_df)

    # --- PLOTTING PER-CELL-STATE BOXPLOT ---
    fig, ax = plt.subplots(figsize=BOXPLOT_FIGSIZE)
    sns.boxplot(
        data=cs_data, x=CONDITION_COL, y="Score", hue="Pathway",
        order=cs_conds, hue_order=TARGET_PATHWAYS, palette=PATHWAY_PALETTE,
        width=BOXPLOT_WIDTH, showfliers=False, boxprops=boxprops, medianprops=medianprops,
        whiskerprops=whiskerprops, capprops=capprops, ax=ax, dodge=True
    )
    sns.stripplot(
        data=cs_data, x=CONDITION_COL, y="Score", hue="Pathway",
        order=cs_conds, hue_order=TARGET_PATHWAYS, dodge=True, 
        palette=["black", "black"], size=STRIP_DOT_SIZE, alpha=STRIP_ALPHA, linewidth=0, legend=False, ax=ax
    )

    for _, row in cs_stats_df.iterrows():
        if row["Significance"] == "": continue
        if row["Condition"] not in cs_conds: continue
        i = cs_conds.index(row["Condition"])

        cond_scores = cs_data[cs_data[CONDITION_COL] == row["Condition"]]["Score"]
        y_range = cond_scores.max() - cond_scores.min() if (cond_scores.max() - cond_scores.min()) > 0 else 1.0
        y = cond_scores.max() + BRACKET_OFFSET_FRAC * y_range
        tick_h = BRACKET_TICK_FRAC * y_range

        x1, x2 = i - 0.2, i + 0.2
        ax.plot([x1, x1, x2, x2], [y, y + tick_h, y + tick_h, y], lw=BRACKET_LINE_WIDTH, color="black")
        ax.text(i, y + tick_h + STAR_GAP_FRAC * y_range, row["Significance"], ha="center", va="bottom", fontsize=STAR_FONT_SIZE, fontweight="bold")

    ax.set_title(f"MP Polarization — {cs}", fontweight="bold", fontsize=8)
    ax.set_xlabel("Condition", fontweight="bold", fontsize=7)
    ax.set_ylabel("Score", fontweight="bold", fontsize=7)
    plt.xticks(rotation=30, ha="right", fontweight="bold", fontsize=7)
    plt.yticks(fontweight="bold", fontsize=7)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[:2], labels[:2], bbox_to_anchor=(1.05, 1), loc="upper left", frameon=False, prop={"weight": "bold", "size": 7}, title="Pathway")
    plt.tight_layout()
    plt.savefig(os.path.join(cs_out, f"{safe_cs}_boxplot.pdf"), dpi=DPI, bbox_inches="tight")
    plt.close()

    # --- PLOTTING PER-CELL-STATE HEATMAP ---
    ht = cs_data.groupby([CONDITION_COL, "Pathway"], observed=True)["Score"].median().unstack("Pathway")
    ht = ht.reindex([c for c in TARGET_CONDITIONS if c in ht.index])
    ht.round(4).to_csv(os.path.join(cs_out, f"{safe_cs}_heatmap_data.csv"))

    fig, ax = plt.subplots(figsize=HEATMAP_FIGSIZE)
    sns.heatmap(ht, cmap=HEATMAP_CMAP, annot=True, fmt=".3f", linewidths=0.5, annot_kws={"weight": "bold", "size": HEATMAP_ANNOT_SIZE}, ax=ax)
    ax.set_title(f"MP Polarization Heatmap — {cs}", fontweight="bold", fontsize=9)
    plt.xticks(rotation=30, ha="right", fontweight="bold", fontsize=7)
    plt.yticks(rotation=0, fontweight="bold", fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(cs_out, f"{safe_cs}_heatmap.pdf"), dpi=DPI, bbox_inches="tight")
    plt.close()

print(f"\n[Per-Cell-State Overview] All sub-state plots and stats successfully saved to {cellstate_dir}")

# Now let's make a combined CSV 

if all_stats:
    combined = pd.concat(all_stats, ignore_index=True)
    combined.to_csv("/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/MP/results/per_cellstate/all_cellstates_polarization_stats.csv", index=False)
    print(f"\nCombined stats saved: /all_cellstates_polarization_stats.csv")




[Per-Cell-State Overview] All sub-state plots and stats successfully saved to /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/MP/results/per_cellstate

Combined stats saved: /all_cellstates_polarization_stats.csv


# Per condition

In [6]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# ------------------------------------------------------------------------------
# 1. STANDALONE CONFIGURATION
# ------------------------------------------------------------------------------
# Assuming adata_mp is already loaded in your environment
RESULTS_DIR = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/MP/results"
FULL_STATS_DIR = os.path.join(RESULTS_DIR, "per_condition_overview")
os.makedirs(FULL_STATS_DIR, exist_ok=True)

SAMPLE_COL     = "Sample_ID"
CONDITION_COL  = "Condition"
CELL_STATE_COL = "cell_state"

ALL_CONDITIONS = [
    "Control",
    "Prematurely-terminated",
    "3month-protocol",
    "6month-protocol",
    "PCMV/PRV",
    "Rejection",
]

TARGET_PATHWAYS = ["Pro-inflammatory", "Anti-inflammatory"]

PATHWAY_PALETTE = {
    "Pro-inflammatory": "#B34DB3",  # Purple
    "Anti-inflammatory": "#4CAF50", # Green
}

BOXPLOT_FIGSIZE      = (4.2, 3.2)
BOXPLOT_WIDTH        = 0.9
BOXPLOT_LINEWIDTH    = 0.9
MEDIAN_LINEWIDTH     = 0.7
STRIP_DOT_SIZE       = 4
STRIP_ALPHA          = 0.6
BRACKET_LINE_WIDTH   = 0.9
BRACKET_OFFSET_FRAC  = 0.02
BRACKET_TICK_FRAC    = 0.001
STAR_GAP_FRAC        = 0.0001
STAR_FONT_SIZE       = 8
DPI                  = 300

print("Extracting full dataset for all conditions...")

# Pull directly from adata_mp to ensure no previous filters apply
df_all = adata_mp.obs[[f"{p}_score" for p in TARGET_PATHWAYS] + [SAMPLE_COL, CONDITION_COL, CELL_STATE_COL]].copy()
df_all.columns = [c.replace("_score", "") for c in df_all.columns]

# Ensure we only process the 6 main conditions
df_all = df_all[df_all[CONDITION_COL].isin(ALL_CONDITIONS)]

# Pseudobulk per (Condition, Cell_State, Sample)
grouped_all = df_all.groupby([CONDITION_COL, CELL_STATE_COL, SAMPLE_COL], observed=True)[TARGET_PATHWAYS].median().reset_index()

# Melt for stat iteration and plotting
melted_all = grouped_all.melt(
    id_vars=[CONDITION_COL, CELL_STATE_COL, SAMPLE_COL],
    value_vars=TARGET_PATHWAYS, var_name="Pathway", value_name="Score"
)
# Loop 
active_all_conds = [c for c in ALL_CONDITIONS if c in melted_all[CONDITION_COL].unique()]
all_condition_stats = []

print("Calculating significance and plotting for all active conditions...")

for cond in active_all_conds:
    # IMPORTANT: Use melted_all, not melted_cs
    cond_data = melted_all[melted_all[CONDITION_COL] == cond].copy()
    if cond_data.empty: continue
        
    safe_cond = cond.replace("/", "_").replace("-", "_").replace(" ", "_")
    cs_list = sorted(cond_data[CELL_STATE_COL].unique())
    n_states = len(cs_list)
    
    cond_stats = []
    cond_pvals = []
    
    for cs in cs_list:
        cs_data = cond_data[cond_data[CELL_STATE_COL] == cs]
        pro  = cs_data[cs_data["Pathway"] == "Pro-inflammatory"]["Score"].dropna()
        anti = cs_data[cs_data["Pathway"] == "Anti-inflammatory"]["Score"].dropna()

        med_pro  = pro.median() if len(pro) > 0 else np.nan
        med_anti = anti.median() if len(anti) > 0 else np.nan
        log2fc   = np.log2(med_pro / med_anti) if med_anti > 0 and med_pro > 0 else np.nan

        pval = np.nan
        if len(pro) >= 2 and len(anti) >= 2:
            try:
                _, pval = mannwhitneyu(pro, anti, alternative="two-sided")
            except Exception:
                pass

        cond_stats.append({
            "Condition": cond, 
            "Cell_State": cs,
            "N_Pro": len(pro), 
            "N_Anti": len(anti),
            "Median_Pro": round(med_pro, 4) if not np.isnan(med_pro) else np.nan,
            "Median_Anti": round(med_anti, 4) if not np.isnan(med_anti) else np.nan,
            "log2FC_Pro_vs_Anti": round(log2fc, 4) if not np.isnan(log2fc) else np.nan,
            "pvalue": pval,
        })
        if not np.isnan(pval):
            cond_pvals.append(pval)

    cond_stats_df = pd.DataFrame(cond_stats)
    cond_stats_df["FDR"] = np.nan

    if cond_pvals:
        valid_mask = cond_stats_df["pvalue"].notna()
        _, fdr_vals, _, _ = multipletests(cond_stats_df.loc[valid_mask, "pvalue"], method="fdr_bh")
        cond_stats_df.loc[valid_mask, "FDR"] = fdr_vals

    cond_stats_df["Significance"] = cond_stats_df["FDR"].apply(
        lambda f: "***" if f < 0.001 else ("**" if f < 0.01 else ("*" if f < 0.05 else "")) if not np.isnan(f) else ""
    )
    
    cond_stats_df.to_csv(os.path.join(FULL_STATS_DIR, f"{safe_cond}_stats.csv"), index=False)
    all_condition_stats.append(cond_stats_df)

    fig, ax = plt.subplots(figsize=(max(6, n_states * 0.8), BOXPLOT_FIGSIZE[1]))
    
    sns.boxplot(
        data=cond_data, x=CELL_STATE_COL, y="Score", hue="Pathway",
        order=cs_list, hue_order=TARGET_PATHWAYS, palette=PATHWAY_PALETTE,
        width=BOXPLOT_WIDTH, showfliers=False, 
        boxprops=dict(edgecolor="black", linewidth=BOXPLOT_LINEWIDTH),
        medianprops=dict(color="black", linewidth=MEDIAN_LINEWIDTH),
        whiskerprops=dict(color="black", linewidth=BOXPLOT_LINEWIDTH),
        capprops=dict(color="black", linewidth=BOXPLOT_LINEWIDTH),
        ax=ax, dodge=True
    )
    
    sns.stripplot(
        data=cond_data, x=CELL_STATE_COL, y="Score", hue="Pathway",
        order=cs_list, hue_order=TARGET_PATHWAYS, dodge=True, 
        palette=["black", "black"], size=STRIP_DOT_SIZE, alpha=STRIP_ALPHA, 
        linewidth=0, legend=False, ax=ax
    )

    # Add significance brackets
    for _, row in cond_stats_df.iterrows():
        if row["Significance"] == "": continue
        if row["Cell_State"] not in cs_list: continue
        i = cs_list.index(row["Cell_State"])

        cs_scores = cond_data[cond_data[CELL_STATE_COL] == row["Cell_State"]]["Score"]
        y_range = cs_scores.max() - cs_scores.min() if (cs_scores.max() - cs_scores.min()) > 0 else 1.0
        y = cs_scores.max() + BRACKET_OFFSET_FRAC * y_range
        tick_h = BRACKET_TICK_FRAC * y_range

        x1, x2 = i - 0.2, i + 0.2
        ax.plot([x1, x1, x2, x2], [y, y + tick_h, y + tick_h, y], lw=BRACKET_LINE_WIDTH, color="black")
        ax.text(i, y + tick_h + STAR_GAP_FRAC * y_range, row["Significance"], ha="center", va="bottom", fontsize=STAR_FONT_SIZE, fontweight="bold")

    ax.set_title(f"MP Polarization — {cond}", fontweight="bold", fontsize=9)
    ax.set_xlabel("Cell State", fontweight="bold", fontsize=8)
    ax.set_ylabel("Score", fontweight="bold", fontsize=8)
    plt.xticks(rotation=90, ha="center", fontweight="bold", fontsize=7)
    plt.yticks(fontweight="bold", fontsize=7)
    
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[:2], labels[:2], bbox_to_anchor=(1.05, 1), loc="upper left", frameon=False,
              prop={"weight": "bold", "size": 7}, title="Pathway")
    
    plt.tight_layout()
    plt.savefig(os.path.join(FULL_STATS_DIR, f"{safe_cond}_all_states.pdf"), dpi=DPI, bbox_inches="tight")
    plt.close()

# Save the combined CSV
if all_condition_stats:
    combined_cond_stats = pd.concat(all_condition_stats, ignore_index=True)
    output_csv = os.path.join(FULL_STATS_DIR, "FULL_all_conditions_polarization_stats.csv")
    combined_cond_stats.to_csv(output_csv, index=False)
    print(f"\nSUCCESS: Master statistics and plots for all conditions saved to:\n{FULL_STATS_DIR}")
    

Extracting full dataset for all conditions...
Calculating significance and plotting for all active conditions...

SUCCESS: Master statistics and plots for all conditions saved to:
/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/MP/results/per_condition_overview
